# 02 · Campaign — BindCraft + RFdiffusion at the conserved epitope

**Standard slot:** *design campaign.* **For Project 07 this means:** run both binder paradigms against
the conserved ACE2-facing epitope and assemble the pool (D2). Diversity before filtering: generate
hundreds, filter later. The real campaign wants an **A100** — develop on `mock` here.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Verify the upstreams still exist (pin commits — they change)

Tools move; pin a commit and confirm the repo/notebook still resolves before a real run.

In [ ]:
import requests
UPSTREAMS = {
    "BindCraft":     "https://github.com/martinpacesa/BindCraft",       # pin a commit
    "FreeBindCraft": "https://github.com/cytokineking/FreeBindCraft",   # free-tier fallback (VERIFY)
    "RFdiffusion":   "https://github.com/RosettaCommons/RFdiffusion",
    "ColabDesign":   "https://github.com/sokrypton/ColabDesign",
    "ColabFold":     "https://github.com/sokrypton/ColabFold",          # AF2-Multimer
}
for name, url in UPSTREAMS.items():
    try:
        r = requests.head(url, timeout=20, allow_redirects=True)
        print(f"{name:14s} {r.status_code}  {url}")
    except Exception as e:
        print(f"{name:14s} ERR  {url}  ({e})")

## Run both paradigms (mock; switch tool= on an A100)

BindCraft (50–200 designs) + RFdiffusion binder mode (500–1000 backbones → ProteinMPNN). Here we use
small mock counts so the notebook runs anywhere. All numbers are **SYNTHETIC**.

In [ ]:
import binder_tools as bt, pandas as pd

TARGET = "RBD"
HOTSPOTS = bt.parse_hotspots("E417,E484,E501")   # EXAMPLE — verify

bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=50, tool="mock")     # -> 50-200 real
rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=200, tool="mock")  # -> 500-1000 real
pool = bc + rf
bt.score_designs(pool, tool="mock")
df = pd.DataFrame([d.as_row() for d in pool])
df.to_csv("results/designs.csv", index=False)
print("pool:", len(pool), "| BindCraft:", len(bc), "| RFdiffusion:", len(rf))
print("wrote results/designs.csv", df.shape, "(all SYNTHETIC / EXAMPLE_DATA)")
df.head(3)

## D2 checklist
- [ ] Full pool from **both** paradigms in `results/designs.csv` (hundreds at real scale).
- [ ] Design log: tool versions, params, seeds, runtimes in `LOG.md`.
- [ ] Interim report (3–4 pages).

**Next:** `03_filter_and_rank.ipynb` — the shared binder filter.